# Phase 4 — Draft Generation (+ Phase 5 prep)

**How to use this:** copy each cell below into your notebook, right after the Phase 3 cells
(after "What Phase 3 Adds", before "Next: Phase 4"). Run in order.

This phase does two things:
1. **Generates actual draft SOP update language** — not just "this is affected," but the
   rewritten paragraph, with citations back to the source policy.
2. **Packages everything into one clean function and a JSON export** — this is what Phase 5
   (the UI) will call. Building it this way now means the UI step is just "call one function
   and display the result," instead of re-wiring your pipeline later.

## Step 1 — Generate a draft SOP update

Given a detected change and the specific SOP it affects, ask the model to write the actual
updated procedure language, citing the new policy.

In [ ]:
def generate_sop_draft(sop, change_summary, new_doc, max_new_tokens=300):
    """Generates a draft rewrite of one internal SOP, reflecting a detected policy change."""
    prompt = f"""You are a Medicaid compliance analyst drafting an SOP update. Rewrite the
procedure below to reflect the policy change. Keep the same structure and tone as the
original, change only what the policy change requires, and end with a citation line noting
the source policy and effective date.

ORIGINAL PROCEDURE [{sop['id']}] {sop['title']}:
{sop['text']}

POLICY CHANGE:
{change_summary}

SOURCE POLICY: {new_doc['title']} (effective {new_doc['metadata']['effective_date']})

Rewritten procedure:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print("generate_sop_draft() ready.")

## Step 2 — Decide which SOPs need a drafted update

Not every affected SOP needs a rewrite (e.g. one might just need a reference number updated
elsewhere). We ask the model to classify each affected SOP as "needs rewrite" or "no rewrite
needed," then only draft the ones that do — this keeps the demo output focused and avoids
wasted generation time.

In [ ]:
import re

def parse_affected_sop_ids(impact_text, sops):
    """Extracts which SOP IDs were flagged as affected from the impact analysis text."""
    affected_ids = []
    for sop in sops:
        if sop["id"] in impact_text:
            affected_ids.append(sop["id"])
    return affected_ids


def needs_rewrite(sop, change_summary, impact_text):
    """Simple heuristic: if the SOP ID appears with 'affected' language near it, draft it.
    (For the demo, we draft every flagged SOP — a production version could add a
    classification step here to skip trivial/no-op changes.)"""
    return sop["id"] in impact_text


print("Helper functions ready.")

## Step 3 — One function that runs the whole pipeline

This is the important part for Phase 5: instead of scattered cells, everything (change
detection -> impact mapping -> draft generation) is wrapped into a single function that
takes a policy family and returns one clean, structured result. **This is the function the
UI will call.**

In [ ]:
def run_full_compliance_check(old_doc, new_doc, sops):
    """
    Runs the entire Phase 3 + Phase 4 pipeline for one policy update and returns a single
    structured dict — this is the function Phase 5's UI will call directly.
    """
    # Step A: detect changes
    changes = detect_changes(old_doc, new_doc)

    # Step B: map impact
    impact_text = find_affected_sops(changes, sops)
    affected_ids = parse_affected_sop_ids(impact_text, sops)

    # Step C: draft updates for affected SOPs
    drafts = []
    for sop in sops:
        if sop["id"] in affected_ids:
            draft_text = generate_sop_draft(sop, changes, new_doc)
            drafts.append({
                "sop_id": sop["id"],
                "sop_title": sop["title"],
                "original_text": sop["text"],
                "draft_text": draft_text,
            })

    result = {
        "policy_family": new_doc["metadata"]["policy_family"],
        "old_version": {
            "title": old_doc["title"],
            "effective_date": old_doc["metadata"]["effective_date"],
        },
        "new_version": {
            "title": new_doc["title"],
            "effective_date": new_doc["metadata"]["effective_date"],
        },
        "changes_summary": changes,
        "impact_analysis": impact_text,
        "affected_sop_ids": affected_ids,
        "drafted_updates": drafts,
        "status": "drafts_ready_for_review",
    }
    return result


print("run_full_compliance_check() ready — this is the main entry point for Phase 5.")

## Step 4 — Run the full pipeline end to end

This replaces the separate Phase 3 loop — one call now produces everything, including drafts.

In [ ]:
# Run the complete pipeline for every detected policy update
full_results = []
for old, new in pairs:
    print(f"Running full pipeline for: {new['title']}...")
    result = run_full_compliance_check(old, new, SAMPLE_INTERNAL_SOPS)
    full_results.append(result)
    print(f"  -> {len(result['drafted_updates'])} draft update(s) generated.")

print("\nDone. Full results stored in `full_results`.")

## Step 5 — Print a demo-ready view

A clean, readable version of the final output — this is what you'd literally show on screen
during a demo.

In [ ]:
def print_full_report(result):
    print("=" * 70)
    print(f"COMPLIANCE UPDATE — {result['policy_family']}")
    print("=" * 70)
    print(f"\n{result['old_version']['title']}  -->  {result['new_version']['title']}")
    print(f"\nWHAT CHANGED:\n{result['changes_summary']}")
    print(f"\nIMPACT ANALYSIS:\n{result['impact_analysis']}")

    for draft in result["drafted_updates"]:
        print(f"\n{'-' * 70}")
        print(f"DRAFT UPDATE — [{draft['sop_id']}] {draft['sop_title']}")
        print(f"{'-' * 70}")
        print(f"\nORIGINAL:\n{draft['original_text']}")
        print(f"\nDRAFT REWRITE:\n{draft['draft_text']}")

    print(f"\nSTATUS: {result['status']}")
    print("=" * 70)


for result in full_results:
    print_full_report(result)

## Step 6 — Export to JSON (this is what Phase 5 needs)

Phase 5 will build a UI that displays this data. Rather than the UI needing to talk to this
notebook directly, we export the structured results to a JSON file. This becomes realistic
sample data to build the UI against immediately, and later can be swapped for a live call to
`run_full_compliance_check()` if you deploy this as a real backend.

In [ ]:
import json

with open("/kaggle/working/compliance_results.json", "w") as f:
    json.dump(full_results, f, indent=2)

print("Saved to /kaggle/working/compliance_results.json")
print(f"Contains {len(full_results)} compliance update record(s).")

# Preview the structure
print("\nPreview of one record's keys:")
print(list(full_results[0].keys()))

---
## What Phase 4 Adds

Your engine now produces the complete flow end to end:
**detect change -> map impact -> draft the actual updated procedure -> package as structured
data.**

This is the full "detect impact -> draft update" demo flow originally scoped for your
agency/investor pitch — nothing left to fake or hand-wave.

## Ready for Phase 5

Two things are now in place specifically to make the UI build easy:
1. **`run_full_compliance_check()`** — one function, clean inputs/outputs, ready to be called
   from a backend if this ever goes beyond a local demo.
2. **`compliance_results.json`** — download this file from `/kaggle/working/`. It's realistic
   structured data we can load directly into a polished web UI, so Phase 5 can start
   immediately without waiting on live model calls every time you tweak the design.

**Next step:** download `compliance_results.json` from your Kaggle output panel, and share it
back — that's what I'll use to build the Phase 5 front-end.